In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir("/content/drive/MyDrive/Final_Year_Project/Result/Datasets")

In [ ]:
import pandas as pd
dq = pd.read_csv('cleaned_combined_dataset.csv')
dq.head(10)

,Filename,cA5_ChromaBin_10_kurtosis,cA5_ChromaBin_10_max,cA5_ChromaBin_10_mean,cA5_ChromaBin_10_min,cA5_ChromaBin_10_ptp,cA5_ChromaBin_10_skew,cA5_ChromaBin_10_std,cA5_ChromaBin_11_kurtosis,cA5_ChromaBin_11_max,...,cD5_SpectralRolloff_std,cD5_Tempo,cD5_ZeroCrossingRate_kurtosis,cD5_ZeroCrossingRate_max,cD5_ZeroCrossingRate_mean,cD5_ZeroCrossingRate_min,cD5_ZeroCrossingRate_ptp,cD5_ZeroCrossingRate_skew,cD5_ZeroCrossingRate_std,Label
0,35,-1.753809,0.324940,0.193574,0.066523,0.258417,0.030279,0.105707,-1.417024,0.683241,...,1267.112825,1,-1.600332,0.444824,0.352783,0.258789,0.186035,-0.025410,0.072825,1
1,41,-1.315409,0.283327,0.188532,0.116862,0.166465,0.406989,0.063592,-1.005179,0.819910,...,144.849810,4,-1.404029,0.451660,0.357666,0.249023,0.202637,-0.214015,0.076827,1
2,37,-1.568067,1.000000,0.774333,0.439647,0.560353,-0.332529,0.234820,-1.000510,1.000000,...,968.440699,4,-1.456866,0.395508,0.316406,0.231445,0.164062,-0.102977,0.062389,1
3,43,-0.883223,0.471236,0.229553,0.078683,0.392553,0.813980,0.146994,-1.083850,0.643118,...,1338.916839,4,-1.211841,0.437012,0.346069,0.241699,0.195312,-0.244085,0.071621,1
4,32,-0.738453,0.279715,0.149517,0.089153,0.190563,1.063048,0.076203,-1.178705,0.511554,...,378.327716,4,-1.469688,0.419922,0.330566,0.246094,0.173828,0.080142,0.066211,1
5,39,-0.770359,0.389507,0.244213,0.167465,0.222042,0.997644,0.085826,-1.235216,0.193349,...,1021.693193,4,-1.425132,0.412109,0.329712,0.245117,0.166992,-0.039630,0.062990,1
6,42,-1.898464,0.275869,0.163570,0.040186,0.235683,-0.048233,0.103462,-1.903719,0.581915,...,1171.471066,4,-1.458414,0.400391,0.321167,0.242676,0.157715,0.013506,0.059854,1
7,36,-0.681142,0.347082,0.174059,0.110190,0.236892,1.137064,0.100155,-0.952784,0.333001,...,1378.072428,4,-1.605701,0.407715,0.323608,0.240234,0.167480,0.010509,0.065635,1
8,44,-1.809050,0.232696,0.126638,0.038084,0.194612,0.121201,0.083063,-1.211408,0.298689,...,227.057902,4,-1.491590,0.461914,0.364990,0.271484,0.190430,0.050111,0.072793,1
9,38,-0.784030,0.217628,0.155339,0.120381,0.097248,0.956168,0.037004,-1.246312,1.000000,...,815.565627,4,-1.668005,0.397949,0.311279,0.244629,0.153320,0.229393,0.063449,1


In [ ]:
!pip install xgboost
!pip install optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import optuna
import seaborn as sns
import matplotlib.pyplot as plt

# Load the cleaned dataset
data = pd.read_csv('cleaned_combined_dataset.csv')

# Separate features and labels
X = data.drop(columns=['Label'])  # Features
y = data['Label']                # Target label

# Handle missing values (if any, as a safeguard)
X.fillna(X.median(), inplace=True)

# Apply SMOTE for balancing
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
    }

    # Perform stratified 10-fold cross-validation
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accuracy_scores = []

    for train_idx, valid_idx in cv.split(X_scaled, y_balanced):
        X_train, X_valid = X_scaled[train_idx], X_scaled[valid_idx]
        y_train, y_valid = y_balanced[train_idx], y_balanced[valid_idx]

        model = XGBClassifier(eval_metric='logloss', random_state=42, **params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_valid)
        accuracy_scores.append(accuracy_score(y_valid, y_pred))

    # Return the average accuracy across folds
    return np.mean(accuracy_scores)

# Optimize hyperparameters using Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train a final model using the best hyperparameters
final_model = XGBClassifier(eval_metric='logloss', random_state=42, **best_params)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_balanced, test_size=0.3, random_state=42, stratify=y_balanced)
final_model.fit(X_train, y_train)

# Predict on the test set
y_pred = final_model.predict(X_test)
conf_matrix = confusion_matrix(y_test, y_pred)

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred) * 100
precision = precision_score(y_test, y_pred) * 100
recall = recall_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100
specificity = (conf_matrix[0, 0] / (conf_matrix[0, 0] + conf_matrix[0, 1])) * 100
g_mean = np.sqrt((recall / 100) * (specificity / 100)) * 100

# Display evaluation metrics
print("Final Model Evaluation (in percentages):")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall: {recall:.2f}%")
print(f"F1 Score: {f1:.2f}%")
print(f"Specificity: {specificity:.2f}%")
print(f"Geometric Mean: {g_mean:.2f}%")

# Confusion Matrix Heatmap
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=['HC', 'PD'], yticklabels=['HC', 'PD'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix Heatmap")
plt.show()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 15.6 MB/s eta 0:00:00


[I 2025-04-08 10:49:42,293] A new study created in memory with name: no-name-7b530273-0357-4774-949e-c60ab84325d3
[I 2025-04-08 10:53:03,562] Trial 0 finished with value: 0.43999999999999995 and parameters: {'n_estimators': 148, 'learning_rate': 0.1459075631436959, 'max_depth': 7, 'colsample_bytree': 0.5817195711795451, 'subsample': 0.6578678618111156, 'min_child_weight': 3, 'gamma': 4.260661919636669}. Best is trial 0 with value: 0.43999999999999995.
[I 2025-04-08 10:56:01,734] Trial 1 finished with value: 0.45636363636363636 and parameters: {'n_estimators': 268, 'learning_rate': 0.21363237732758023, 'max_depth': 10, 'colsample_bytree': 0.9466268825478805, 'subsample': 0.8283255145433421, 'min_child_weight': 7, 'gamma': 3.6840361400151633}. Best is trial 1 with value: 0.45636363636363636.
[I 2025-04-08 11:02:05,298] Trial 2 finished with value: 0.45 and parameters: {'n_estimators': 131, 'learning_rate': 0.048789191639528384, 'max_depth': 10, 'colsample_bytree': 0.8842830086842919, 'su